# Generalized Linear Models

Notebook for **Chapter 8** of the *stats* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/stats/ch08.html

Tip: change a number, predict the output, then run the cell again.

## 8.1 The GLM Idea and Link Functions

## 8.2 Logistic Regression and Odds Ratios

## 8.3 Poisson Regression for Counts

## 8.4 Overdispersion and the Negative Binomial Model

## 8.5 Choosing the Right GLM

## 8.6 Rates and Exposure: Offsets Done Properly

Section 8.3 mentioned the offset in one line. It deserves a full example, because forgetting it silently flips conclusions. A count is only comparable between rows if every row had the same chance to produce events. If one site was watched for 5 days and another for 60, raw counts say more about the watching than about the sites. The fix is to model the *rate* (events per unit of exposure) and let the model handle the exposure.

In [ ]:
import numpy as np
rng = np.random.default_rng(8)
n = 300
treat = rng.integers(0, 2, n)
days = np.where(treat == 1, rng.uniform(5, 25, n), rng.uniform(20, 60, n))   # treated sites watched for fewer days
rate = np.exp(-1.2 + 0.4 * treat)                                             # true events per day
y = rng.poisson(rate * days)

def poisson_irls(X, y, offset, iters=25):
    b = np.zeros(X.shape[1]); b[0] = np.log(y.mean() + 0.1)
    for _ in range(iters):
        eta = X @ b + offset; mu = np.exp(eta)
        W = mu; z = eta - offset + (y - mu) / mu
        XtW = X.T * W
        b = np.linalg.solve(XtW @ X, XtW @ z)
    mu = np.exp(X @ b + offset)
    se = np.sqrt(np.diag(np.linalg.inv((X.T * mu) @ X)))
    return b, se
X = np.column_stack([np.ones(n), treat])
b0, s0 = poisson_irls(X, y, np.zeros(n))
b1, s1 = poisson_irls(X, y, np.log(days))
print("true treatment effect on log rate: 0.400")
print(f"no offset : coef {b0[1]:.3f} (SE {s0[1]:.3f})  rate ratio {np.exp(b0[1]):.2f}")
print(f"offset    : coef {b1[1]:.3f} (SE {s1[1]:.3f})  rate ratio {np.exp(b1[1]):.2f}")
print(f"mean days: control {days[treat==0].mean():.1f}, treated {days[treat==1].mean():.1f}")
print(f"baseline rate per day (offset model): {np.exp(b1[0]):.3f}  (true {np.exp(-1.2):.3f})")

true treatment effect on log rate: 0.400
no offset : coef -0.539 (SE 0.039)  rate ratio 0.58
offset    : coef 0.413 (SE 0.039)  rate ratio 1.51
mean days: control 39.9, treated 15.4
baseline rate per day (offset model): 0.307  (true 0.301)


## 8.7 Excess Zeros: Zero-Inflated and Hurdle Models

Many count variables have more zeros than any Poisson or negative binomial can explain: purchases per customer (many people never buy), claims per policy, sick days per employee. The usual story is two groups mixed together. Some units are **structural zeros** (they cannot produce events, such as never-buyers) and the rest follow an ordinary count process that also produces some zeros by chance.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import gammaln
rng = np.random.default_rng(81)
n = 2000
structural = rng.random(n) < 0.30          # never-buyers
y = np.where(structural, 0, rng.poisson(3.0, n))
print("share of zeros observed:", round((y == 0).mean(), 3), " mean:", round(y.mean(), 3), " variance:", round(y.var(), 3))
lam_p = y.mean()
print("Poisson with same mean predicts zeros:", round(np.exp(-lam_p), 3))

def nll(p):
    pi = 1 / (1 + np.exp(-p[0])); lam = np.exp(p[1])
    ll0 = np.log(pi + (1 - pi) * np.exp(-lam))
    llp = np.log(1 - pi) - lam + y * np.log(lam) - gammaln(y + 1)
    return -np.where(y == 0, ll0, llp).sum()
r = minimize(nll, [0.0, 0.5], method="Nelder-Mead", options={"xatol": 1e-8, "fatol": 1e-10})
pi, lam = 1 / (1 + np.exp(-r.x[0])), np.exp(r.x[1])
ll_pois = (-lam_p + y * np.log(lam_p) - gammaln(y + 1)).sum()
print(f"ZIP estimates: pi = {pi:.3f} (true 0.30), lambda = {lam:.3f} (true 3.00)")
print(f"AIC Poisson {2*1 - 2*ll_pois:.1f}   AIC ZIP {2*2 + 2*r.fun:.1f}")

share of zeros observed: 0.344  mean: 2.09  variance: 4.086
Poisson with same mean predicts zeros: 0.124
ZIP estimates: pi = 0.311 (true 0.30), lambda = 3.032 (true 3.00)
AIC Poisson 8464.9   AIC ZIP 7397.4


## 8.8 Ordinal and Multinomial Outcomes

Logistic regression handles two outcomes. Real labels often have more. Two cases need different tools and mixing them up wastes information or breaks assumptions.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(82)
n = 1500
x = rng.normal(0, 1, n)
# proportional odds: logit P(y <= k) = theta_k - beta x, with theta = (-0.5, 1.2), beta = 0.9
th_true, beta_true = np.array([-0.5, 1.2]), 0.9
cum = 1 / (1 + np.exp(-(th_true[None, :] - beta_true * x[:, None])))
u = rng.random(n)
y = (u > cum[:, 0]).astype(int) + (u > cum[:, 1]).astype(int)       # 0 low, 1 mid, 2 high
print("class counts (low, mid, high):", np.bincount(y))
def nll(p):
    t1, d, b = p[0], np.exp(p[1]), p[2]; t2 = t1 + d
    c1 = 1 / (1 + np.exp(-(t1 - b * x))); c2 = 1 / (1 + np.exp(-(t2 - b * x)))
    pr = np.where(y == 0, c1, np.where(y == 1, c2 - c1, 1 - c2))
    return -np.log(np.clip(pr, 1e-12, None)).sum()
r = minimize(nll, [0, 0, 0], method="BFGS")
t1, t2, b = r.x[0], r.x[0] + np.exp(r.x[1]), r.x[2]
print(f"thresholds {t1:.2f}, {t2:.2f} (true -0.50, 1.20); beta {b:.2f} (true 0.90); OR per SD of x: {np.exp(b):.2f}")
c1 = 1/(1+np.exp(-(t1 - b*1.0))); c2 = 1/(1+np.exp(-(t2 - b*1.0)))
print(f"at x = +1 SD: P(low) {c1:.3f}  P(mid) {c2-c1:.3f}  P(high) {1-c2:.3f}")
# multinomial for comparison (unordered): 3 churn reasons from 2 features
rng = np.random.default_rng(83)
m = 1200; F = rng.normal(0, 1, (m, 2))
logits = np.column_stack([np.zeros(m), 0.8 * F[:, 0] - 0.2, -0.9 * F[:, 1] + 0.1])
P = np.exp(logits); P /= P.sum(1, keepdims=True)
cls = np.array([rng.choice(3, p=p) for p in P])
clf = LogisticRegression(C=1e6, max_iter=2000).fit(F, cls)
co = clf.coef_ - clf.coef_[0]           # relative to class 0 as the reference
print("multinomial coefficients vs reference class 0 (true: class1 [0.8, 0], class2 [0, -0.9]):")
print(np.round(co[1:], 2))
print("predicted probabilities at (1, 0):", np.round(clf.predict_proba([[1.0, 0.0]])[0], 3))

class counts (low, mid, high): [565 520 415]
thresholds -0.57, 1.11 (true -0.50, 1.20); beta 0.83 (true 0.90); OR per SD of x: 2.30
at x = +1 SD: P(low) 0.198  P(mid) 0.370  P(high) 0.432
multinomial coefficients vs reference class 0 (true: class1 [0.8, 0], class2 [0, -0.9]):
[[ 0.65  0.06]
 [-0.07 -0.87]]
predicted probabilities at (1, 0): [0.27  0.461 0.269]


## 8.9 Judging a Logistic Model: AUC, Calibration and Separation

A logistic model can fail in two different ways: it can rank cases badly, or it can rank them well but give the wrong probabilities. They need different checks.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from scipy.stats import chi2
rng = np.random.default_rng(84)
n = 2000
X = rng.normal(0, 1, (n, 2))
p = 1 / (1 + np.exp(-(-1.0 + 1.1 * X[:, 0] + 0.6 * X[:, 1])))
y = (rng.random(n) < p).astype(int)
tr, te = slice(0, 1200), slice(1200, n)
m = LogisticRegression(C=1e6).fit(X[tr], y[tr])
ph = m.predict_proba(X[te])[:, 1]; yt = y[te]
def auc(s, yy):
    r = s.argsort().argsort() + 1
    n1 = yy.sum(); n0 = len(yy) - n1
    return (r[yy == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
print("test AUC:", round(auc(ph, yt), 3))
# Hosmer-Lemeshow on the training data (g - 2 = 8 df), 10 equal-count groups
ptr = m.predict_proba(X[tr])[:, 1]; ytr = y[tr]
o = np.argsort(ptr); groups = np.array_split(o, 10)
hl = 0
for g in groups:
    n_g, obs, exp = len(g), ytr[g].sum(), ptr[g].sum()
    pbar = exp / n_g
    hl += (obs - exp) ** 2 / (n_g * pbar * (1 - pbar))
print(f"Hosmer-Lemeshow chi2 = {hl:.2f} on 8 df, p = {1 - chi2.cdf(hl, 8):.3f}")
print("decile  mean predicted  observed rate")
for i, g in enumerate(groups):
    print(f"{i+1:>6}  {ptr[g].mean():>14.3f}  {ytr[g].mean():>13.3f}")
# separation
xs = np.concatenate([rng.normal(-2, .3, 50), rng.normal(2, .3, 50)]).reshape(-1, 1); ys = np.r_[np.zeros(50), np.ones(50)]
print("perfect separation, slope as the penalty weakens:")
for C in [1, 100, 1e4, 1e6]:
    print(f"  C = {C:>9g}: slope = {LogisticRegression(C=C, max_iter=5000).fit(xs, ys).coef_[0][0]:.2f}")

test AUC: 0.784
Hosmer-Lemeshow chi2 = 8.84 on 8 df, p = 0.356
decile  mean predicted  observed rate
     1           0.033          0.025
     2           0.078          0.075
     3           0.126          0.150
     4           0.180          0.158
     5           0.235          0.217
     6           0.298          0.367
     7           0.372          0.308
     8           0.468          0.525
     9           0.602          0.558
    10           0.791          0.800
perfect separation, slope as the penalty weakens:
  C =         1: slope = 2.32
  C =       100: slope = 4.68
  C =     10000: slope = 5.98
  C =     1e+06: slope = 6.01


## Chapter recap

## Practice set

**1 (easy).** p = 0.75. Give odds and log-odds. **Answer:** Odds = 0.75/0.25 = 3. Log-odds = ln 3 = 1.099.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/stats/ch08.html